# 00 · Setup do Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SEU_USUARIO/waid-segmentation/blob/main/notebooks/00_setup_colab.ipynb)

Prepara o ambiente: **monta o Drive → clona/atualiza o repositório → instala dependências → verifica a GPU**.

**Antes de rodar:**
1. `Ambiente de execução` → `Alterar tipo de ambiente` → **GPU**.
2. (Só para `git push`) crie o segredo `GITHUB_TOKEN` em **Secrets** (ícone de chave) e ative o acesso ao notebook.
3. Ajuste `GITHUB_USER` na célula de configuração.

## 1. Configuração

In [ ]:
GITHUB_USER = "SEU_USUARIO"        # <- ajuste
REPO_NAME   = "waid-segmentation"
BRANCH      = "main"

REPO_URL  = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
REPO_DIR  = f"/content/{REPO_NAME}"            # efêmero (some ao desconectar)
DRIVE_DIR = "/content/drive/MyDrive/waid"      # persistente (dados e checkpoints)

## 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
for sub in ["data/raw", "data/processed", "checkpoints", "outputs"]:
    os.makedirs(f"{DRIVE_DIR}/{sub}", exist_ok=True)
print("Pastas no Drive prontas em:", DRIVE_DIR)

## 3. Clonar / atualizar o repositório

O token (se existir) fica só no `.git/config` do ambiente temporário `/content` — nunca é escrito no notebook nem no Drive.

In [ ]:
import os, subprocess

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
    print("GITHUB_TOKEN não encontrado: clone somente leitura (repo público).")

auth_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone -b {BRANCH} {auth_url} {REPO_DIR}

# identidade para commits feitos a partir do Colab
!git -C {REPO_DIR} config user.name  "{GITHUB_USER}"
!git -C {REPO_DIR} config user.email "{GITHUB_USER}@users.noreply.github.com"

%cd {REPO_DIR}

## 4. Ligar dados e checkpoints do Drive ao repositório

Links simbólicos: o código enxerga `data/` e `checkpoints/` como pastas normais, mas os arquivos ficam no Drive (e o `.gitignore` os mantém fora do Git).

In [ ]:
for name, target in [("data", "data"), ("checkpoints", "checkpoints"), ("outputs", "outputs")]:
    link = f"{REPO_DIR}/{name}"
    if not os.path.lexists(link):
        os.symlink(f"{DRIVE_DIR}/{target}", link)
!ls -la {REPO_DIR} | grep -E "data|checkpoints|outputs"

## 5. Dependências

In [ ]:
# PyTorch/torchvision já vêm no Colab; instalamos só o restante.
!pip install -q -r requirements.txt

## 6. Verificação do ambiente

In [ ]:
import sys, torch
sys.path.insert(0, REPO_DIR)   # permite `import src...`

print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA   :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU    :", torch.cuda.get_device_name(0))
else:
    print("⚠️ Sem GPU. Ative em: Ambiente de execução > Alterar tipo de ambiente.")

## 7. Dados do WAID

Os CSVs **não** estão no repositório. Baixe-os de <https://github.com/petrobras/waid> para `MyDrive/waid/data/raw/` (basta fazer uma vez; ficam salvos no Drive).
O script de download será adicionado em `scripts/` na Fase 1.

## 8. Salvar alterações (opcional)

Edite o código no VS Code/github.dev e use `git pull` aqui. Se precisar enviar algo a partir do Colab:

In [ ]:
# !git add -A
# !git commit -m "mensagem clara do que mudou"
# !git push origin {BRANCH}